In [1]:
import json

# load results

In [2]:
results_folder = 'eval_output/complete/medmcqa_answeris'
model_size = 'llama-1B-20BT'

file = f'{results_folder}/{model_size}-weightdecay0.001-seed42-medmcqa--greedy/medmcqa/chunk_0-1/grader_results.json'
with open(file, 'r') as f:
    wd_low_greedy = json.load(f)

file = f'{results_folder}/{model_size}-weightdecay0.001-seed42-medmcqa--n16/medmcqa/chunk_0-1/grader_results.json'
with open(file, 'r') as f:
    wd_low_n16 = json.load(f)


file = f'{results_folder}/{model_size}-weightdecay1.0-seed42-medmcqa--greedy/medmcqa/chunk_0-1/grader_results.json'
with open(file, 'r') as f:
    wd_high_greedy = json.load(f)

file = f'{results_folder}/{model_size}-weightdecay1.0-seed42-medmcqa--n16/medmcqa/chunk_0-1/grader_results.json'
with open(file, 'r') as f:
    wd_high_n16 = json.load(f)


# look at distribution of choices for correct answers

In [3]:
#look at true correct answers, can use any of the results
n_A = 0
n_B = 0
n_C = 0
n_D = 0

results = wd_low_greedy['updated_examinee_results']

for i in range(len(results)):
    generation_dict = results[i]
    if generation_dict['gt_solution'] == 'A':
        n_A += 1
    elif generation_dict['gt_solution'] == 'B':
        n_B += 1
    elif generation_dict['gt_solution'] == 'C':
        n_C += 1
    elif generation_dict['gt_solution'] == 'D':
        n_D += 1

print(f'n_A: {n_A} / {len(results)} ({n_A/len(results)*100:.2f}%)')
print(f'n_B: {n_B} / {len(results)} ({n_B/len(results)*100:.2f}%)')
print(f'n_C: {n_C} / {len(results)} ({n_C/len(results)*100:.2f}%)')
print(f'n_D: {n_D} / {len(results)} ({n_D/len(results)*100:.2f}%)')



n_A: 1348 / 4183 (32.23%)
n_B: 1085 / 4183 (25.94%)
n_C: 925 / 4183 (22.11%)
n_D: 825 / 4183 (19.72%)


# look at generations with incorrect answers

In [4]:
# results = wd_low_greedy['updated_examinee_results'] #list of dicts
results = wd_high_greedy['updated_examinee_results'] #list of dicts
results = wd_low_n16['updated_examinee_results'] #list of dicts
results = wd_high_n16['updated_examinee_results'] #list of dicts

In [5]:
#collect wrong answers

wrong_answers = []
n_wrong_answers = 0

for i in range(len(results)):
    generation_dict = results[i]

    if generation_dict['correct_answer_existence'] is False:
        d = {
            'id': generation_dict['id'],
            'problem': generation_dict['problem'],
            'gt_solution': generation_dict['gt_solution'],
            # 'majority_model_answer': generation_dict['majority_model_answer'],
            'model_responses': generation_dict['model_responses'],
        }
        wrong_answers.append(d)
        n_wrong_answers += 1

print('# wrong answers:', n_wrong_answers)

# wrong answers: 161


In [6]:
#print wrong answers

#for greedy results
# for i in range(len(wrong_answers)):
#     if i % 100 == 0:
#         print(f'----------- {i} / {len(wrong_answers)}')
#         print('Question:', generation_dict['problem'])
#         print('Model response:', wrong_answers[i]['model_responses'][0])
#         print('Correct answer:', wrong_answers[i]['gt_solution'])
#         print('Extracted answer:', wrong_answers[i]['majority_model_answer'])
        

# for greedy or n16 results
for i in range(len(wrong_answers)):
    if i % 100 == 0:
        print(f'*************** {i} / {len(wrong_answers)}')
        print('Question:', generation_dict['problem'])
        print('Correct answer:', wrong_answers[i]['gt_solution'])
        
        for j, r in enumerate(wrong_answers[i]['model_responses']):
            print(f'---> Model response (n={j}):', r['text'])
            print('Extracted answer:', r['extracted_answer'])
            print('')

*************** 0 / 161
Question: Most common type of Non-Hodgkin's lymphoma in the orbit:
Options:
A. B-cell
B. T-cell
C. NK-cell
D. Plasma cell
Correct answer: A
---> Model response (n=0):  The question asks about the concentration of tropicamide in Sarin Antitoxins (SAR). Sarin Antitoxin is a vaccine used to protect against brachial neuronitis caused by the varicella-zoster virus, which causes chickenpox and shingles.

To determine the correct answer, let's examine the options:

A. 0.01: This concentration of tropicamide is not commonly used for vaccine manufacturing reasons.

B. 0.02: This concentration of tropicamide is not typically used for immunization.

C. 0.03: While this concentration might come from a some vaccine development process, it is not the standard concentration for Tropixine.

D. 0.04: According to the explanation, Tropixine is derived from aluminium hydroxide and has an anticoagulant effect. Therefore, it should be converted into its coagulating agent, aluminium 

# look at generations with extracted answer = None

In [7]:
# results = wd_low_greedy['updated_examinee_results'] #list of dicts
results = wd_high_greedy['updated_examinee_results'] #list of dicts
results = wd_low_n16['updated_examinee_results'] #list of dicts
# results = wd_high_n16['updated_examinee_results'] #list of dicts

In [8]:
#collect none answers

none_answers = []
n_none_answers_question_level = 0
n_none_answers_generation_level = 0

for i in range(len(results)):
    generation_dict = results[i]

    #collect all extracted answers for this question
    extracted_answers_lst = [r_dict['extracted_answer'] for r_dict in results[i]['model_responses']]

    # if any extracted answer is None, increment question level counter + collect actual responses
    if any(item is None for item in extracted_answers_lst):
        n_none_answers_question_level += 1
    
        #go through each model generation
        for j, r in enumerate(generation_dict['model_responses']):
            
            if r['extracted_answer'] is None:
                d = {
                    'id': generation_dict['id'], #problem id
                    'generation_number': j, #generation id
                    'problem': generation_dict['problem'],
                    'gt_solution': generation_dict['gt_solution'],
                    'model_response': r['text'],
                    'extracted_answer': r['extracted_answer'],
                }
                none_answers.append(d)
                n_none_answers_generation_level += 1


print('# none answers:')
n_questions = len(results)
n_generations_per_question = len(results[0]['model_responses'])
n_generations = n_questions * n_generations_per_question
print(f'{n_none_answers_question_level} / {n_questions} questions')
print(f'{n_none_answers_generation_level} / {n_generations} generations')

# none answers:
2271 / 4183 questions
4331 / 66928 generations


In [9]:
#print none answers


for i in range(len(none_answers)):
    if i % 200 == 0:
        print(f'*************** {i} / {len(none_answers)}')
        print('Question:', generation_dict['problem'])
        print('Correct answer:', none_answers[i]['gt_solution'])

        print('Generation number (based on order of generation):', none_answers[i]['generation_number'])
        print('Model response:', none_answers[i]['model_response'])
        print('Extracted answer:', none_answers[i]['extracted_answer'])
        
        # for j, r in enumerate(none_answers[i]['model_responses']):
        #     print(f'---> Model response (n={j}):', r['text'])
        #     print('Extracted answer:', r['extracted_answer'])
        #     print('')

*************** 0 / 4331
Question: Most common type of Non-Hodgkin's lymphoma in the orbit:
Options:
A. B-cell
B. T-cell
C. NK-cell
D. Plasma cell
Correct answer: C
Generation number (based on order of generation): 0
Model response:  The question is asking about the best advice for a 29-year-old woman who has a 10-year-old son with Down syndrome, which until now has been identified by the age of 16 weeks as her patient's pregnancy.

To determine the correct answer, let's analyze each option:

A. No test is required now as her age is below 35 years: This statement is incorrect. The American College of Obstetricians and Gynecologists (ACOG) recommends that women with a 10-year history of pregnancy or childbirth outside the normal range (e.g., 35 to 40 weeks of gestation or beyond) should be offered diagnostic testing to rule out genetic disorders or chromosomal abnormalities. Therefore, this statement is incorrect.

B. Ultra sound at this point of time will certainly tell her that next b

# below = scratch

in the following json file of evaluations of model responses, do you see anything potentially incorrect or problematic in 'updated_examinee_results'? here is some info aboutt the entries:
- 'gt_solution' (is the same as 'gt_answer') and is the true correct answer
- 'model_responses' are the raw model outputs. here there is only one response per model
- 'majority_model_answer' is the multiple choice answer extracted from the model's response